# YAML Pipeline Demo

This notebook demonstrates the **`swe.cli.pipeline`** module for defining and
executing multi-step seismic workflows from YAML configuration files.

The pipeline system:
- Reads a YAML config with ordered steps
- Each step calls a Python function with keyword arguments
- Steps can reference outputs from prior steps via `${step_name.key}` syntax
- Steps can be skipped at execution time

In [ ]:
from __future__ import annotations

import logging
from pathlib import Path

from swe.cli.pipeline import Pipeline, PipelineStep

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

EXAMPLES_DIR = Path("../examples")

## 1. Inspect the Example Pipeline Config

The example YAML file ships with SWE at `examples/pipeline_config.yml`.

In [ ]:
config_path = EXAMPLES_DIR / "pipeline_config.yml"
print(config_path.read_text())

## 2. Load the Pipeline

In [ ]:
pipeline = Pipeline.from_yaml(config_path)

print(f"Pipeline has {len(pipeline.steps)} steps:\n")
for step in pipeline.steps:
    print(f"  {step.name}")
    print(f"    function: {step.function}")
    print(f"    params:   {step.params}")
    print()

## 3. Build a Custom Pipeline Programmatically

Pipelines can also be built in Python without YAML.

In [ ]:
custom_pipeline = Pipeline(steps=[
    PipelineStep(
        name="get_cwd",
        function="os.getcwd",
        params={},
    ),
    PipelineStep(
        name="check_exists",
        function="os.path.exists",
        params={"path": "."},
    ),
])

outputs = custom_pipeline.execute()

print("Step outputs:")
for name, result in outputs.items():
    print(f"  {name}: {result}")

## 4. Parameter References (`${step.key}`)

Steps can reference outputs from prior steps. When a step returns a dict
or named tuple, downstream steps can access specific fields.

In [ ]:
# Write a YAML config with parameter references
ref_config = Path("data") / "ref_pipeline.yml"
ref_config.parent.mkdir(parents=True, exist_ok=True)

ref_config.write_text("""
steps:
  - name: get_cwd
    function: os.getcwd
    params: {}
""")

pipeline = Pipeline.from_yaml(ref_config)
outputs = pipeline.execute()
print(f"Current directory: {outputs['get_cwd']}")

## 5. Skipping Steps

Steps can be skipped at execution time by passing their names.

In [ ]:
pipeline = Pipeline(steps=[
    PipelineStep(name="step1", function="os.getcwd", params={}),
    PipelineStep(name="step2", function="os.getcwd", params={}),
    PipelineStep(name="step3", function="os.getcwd", params={}),
])

outputs = pipeline.execute(skip={"step2"})

print(f"Executed steps: {list(outputs.keys())}")
assert "step1" in outputs
assert "step2" not in outputs  # skipped
assert "step3" in outputs
print("Step 2 successfully skipped!")

## 6. CLI Usage

The same pipeline can be run from the command line:

```bash
# Run full pipeline
swe run examples/pipeline_config.yml

# Skip specific steps
swe run examples/pipeline_config.yml --skip download_responses

# Verbose output
swe -vv run examples/pipeline_config.yml
```

## Summary

| Feature | Description |
|---------|-------------|
| YAML config | Declarative pipeline definition |
| `Pipeline.from_yaml()` | Load from file |
| `Pipeline.execute()` | Run all steps sequentially |
| `${step.key}` references | Chain step outputs |
| `--skip` | Skip individual steps |
| `swe run` | CLI entry point |

Pipelines can call any importable Python function, making them extensible
to custom processing steps beyond the built-in SWE modules.